# 64. 봉인 후 추가37개 정답 개봉·사전 지정 모델 평가
63번 예측과프로토콜해시를확인한뒤에만37개정답을연다. 원본Y/0=정상,N/1+가스사유=가스,그외불량/사유불명확=별도계수. 가스가0개이면F1/가스탐지개선은평가불가로표시한다. 개봉뒤모델을튜닝해다시독립검증이라부르지않는다.


In [ ]:
from pathlib import Path
import json,hashlib
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
pred_path=tables/'rg3_independent37_predictions_SEALED.csv'
assert pred_path.exists(),'Predictions must be sealed before labels are opened.'
seal=pd.read_csv(tables/'rg3_independent37_prediction_seal.csv').iloc[0]
assert hashlib.sha256(pred_path.read_bytes()).hexdigest()==seal.prediction_sha256
assert hashlib.sha256((root/'reports/rg3_independent37_locked_protocol.json').read_bytes()).hexdigest()==seal.protocol_sha256
p=pd.read_csv(pred_path,parse_dates=['TimeStamp'])
# FIRST deliberate opening of extra37 quality labels.
raw=pd.read_csv(root/'data/original/labeled_data.csv',usecols=['EQUIP_CD','PART_NAME','TimeStamp','PassOrFail','Reason'])
raw.TimeStamp=pd.to_datetime(raw.TimeStamp)
raw=raw[raw.PART_NAME.str.contains('RG3',na=False)&raw.PART_NAME.str.contains('RH',na=False)]
g=p.merge(raw,on=['EQUIP_CD','PART_NAME','TimeStamp'],validate='one_to_one')
assert len(g)==37
quality=g.PassOrFail.astype(str).str.strip().str.upper()
good=quality.isin(['Y','0','0.0'])
bad=quality.isin(['N','1','1.0'])
gas=bad&g.Reason.fillna('').str.contains('가스')
other=bad&~gas
g['gas_binary_eligible']=good|gas
g['actual_gas']=np.where(gas,1,np.where(good,0,np.nan))
g['label_status']=np.select([good,gas,other],['normal','gas','other_or_unspecified_defect'],default='unknown_quality_label')
rows=[]
for name in ['reference','primary','secondary']:
 selected=g[g.gas_binary_eligible].copy()
 y=selected.actual_gas.to_numpy(int)
 pr=selected[name+'_prediction'].to_numpy(int)
 tn,fp,fn,tp=confusion_matrix(y,pr,labels=[0,1]).ravel()
 gas_count=int(y.sum())
 rows.append(dict(model=name,test_rows=len(g),eligible_gas_normal_rows=len(selected),gas=gas_count,normal=int((y==0).sum()),other_or_unknown_rows=int((~g.gas_binary_eligible).sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if gas_count>0 else np.nan,recall=tp/(tp+fn) if gas_count>0 else np.nan,precision=tp/(tp+fp) if tp+fp>0 else np.nan,can_evaluate_gas_improvement=bool(gas_count>0)))
m=pd.DataFrame(rows)
g.to_csv(tables/'rg3_independent37_unblinded_results.csv',index=False,encoding='utf-8-sig')
m.to_csv(tables/'rg3_independent37_metrics.csv',index=False,encoding='utf-8-sig')
print('BLIND HOLDOUT label status')
print(g.groupby(['label_status','Reason'],dropna=False).size().to_string())
print('PREDECLARED models, no winner selection')
print(m.to_string(index=False))
print('This holdout is consumed after this evaluation. Do not tune on these37 and call a second run independent.')
